# 01 — Training Data Setup (Phase 05 Initialization)

**Status:** `NOT RUN` (Execute this notebook first)

Loads `data/processed/charity-patient-realism-revised.csv`, screens target leakage, assigns `dup_group_id` to exact duplicate rows, and creates a **group-aware stratified** 80/20 train/test split (`StratifiedGroupKFold`) saved to `models/phase_05/`.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "data").exists() and (p / "experiments").exists():
            return p
    return cur

PROJECT_ROOT = find_project_root()
DATA_CSV = PROJECT_ROOT / "data" / "processed" / "charity-patient-realism-revised.csv"
MODELS_DIR = PROJECT_ROOT / "models" / "phase_05"
REPORTS_DIR = PROJECT_ROOT / "reports" / "phase_05"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_CSV)
TARGET_COL = "eligibility"
EXCLUDE_COLS = {
    TARGET_COL, "address", "philhealth_no", "doctor", "ap_consultant", "referred_by",
    "follow_up_date", "date_available", "date_registered", "quotation_particular",
    "quotation_amount", "quotation_remarks", "other_sources_of_financial_support",
    "interview_responses", "other_eligibility_certifications", "estimated_treatment_expenses",
    "approved_financial_assistance", "remaining_medical_expenses", "case_study_narrative",
    "social_economic_assessment", "assessment_notes", "social_worker_assessment_rating",
    "relevant_social_worker_findings", "status", "existing_financial_assistance",
}
FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]

# Group IDs for exact duplicates so identical rows never cross train/test or CV folds
row_hashes = pd.util.hash_pandas_object(df[FEATURE_COLS], index=False)
dup_group_id = row_hashes.astype(str)

y = (df[TARGET_COL].astype(str).str.upper() == "ELIGIBLE").astype(int)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df[FEATURE_COLS], y, groups=dup_group_id))

split_info = {
    "dataset": str(DATA_CSV.relative_to(PROJECT_ROOT)),
    "n_rows": len(df),
    "n_features": len(FEATURE_COLS),
    "feature_cols": FEATURE_COLS,
    "train_indices": train_idx.tolist(),
    "test_indices": test_idx.tolist(),
    "random_seed": 42,
}
(MODELS_DIR / "phase05_split_and_schema.json").write_text(json.dumps(split_info, indent=2), encoding="utf-8")
print(f"Saved split & schema: {len(train_idx)} train, {len(test_idx)} untouched test, {len(FEATURE_COLS)} features.")
